# Optimizers

> bioMONAI optimizers

This section exposes the core optimization components integrated within bioMONAI, wrapping standard optimization structures (such as `OptimWrapper` and `Adam`) to streamline model training workflows.

In [ ]:
#| default_exp optimizers

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

from torch import randn as torchrandn

In [ ]:
#| export

# =================================
# Scientific / data
# =================================
# import numpy as np
# import pandas as pd

# =================================
# PyTorch
# =================================
import torch.optim as toptim

# =================================
# fastai
# =================================
from fastai.optimizer import Adam, OptimWrapper, Optimizer

# =================================
# bioMONAI
# =================================
from bioMONAI.utils import *

## Base Functions
### Backends

In [ ]:
#| export   

OPTIMIZER_BACKENDS = {}


def register_optimizer_backend(name):
    """Register an optimizer backend."""
    def decorator(cls):
        OPTIMIZER_BACKENDS[name] = cls
        return cls
    return decorator



In [ ]:
#| export
OptimWrapper = OptimWrapper

### `OptimWrapper` Class Attributes & Parameter Reference

*A wrapper class for existing PyTorch optimizers to integrate them seamlessly into fastai workflows.*

| Parameter / Attribute | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`params`** | `Tensor \| Iterable` | `None` | Model parameters. Don't set if using a built optimizer. |
| **`opt`** | `Callable \| torch.optim.Optimizer` | `None` | A torch optimizer constructor, or an already built optimizer. |
| **`hp_map`** | `dict` | `None` | A dictionary converting PyTorch optimizer keys to fastai's `Optimizer` keys. Defaults to `pytorch_hp_map`. |
| **`convert_groups`** | `bool` | `True` | Convert parameter groups from splitter or pass unaltered to `opt`. |
| **`**kwargs`** | `VAR_KEYWORD` | – | Additional keyword arguments passed to the optimizer. |

In [ ]:
show_doc(OptimWrapper)

---

[source](https://github.com/fastai/fastai/blob/main/fastai/optimizer.py#LNone){target="_blank" style="float:right; font-size:smaller"}

### OptimWrapper

```python

def OptimWrapper(
    params:Tensor | Iterable=None, # Model parameters. Don't set if using a built optimizer
    opt:Callable | torch.optim.Optimizer=None, # A torch optimizer constructor, or an already built optimizer
    hp_map:dict=None, # A dictionary converting PyTorch optimizer keys to fastai's `Optimizer` keys. Defaults to `pytorch_hp_map`
    convert_groups:bool=True, # Convert parameter groups from splitter or pass unaltered to `opt`
    kwargs:VAR_KEYWORD
):


```

*A wrapper class for existing PyTorch optimizers*

### Backend abstractions

In [ ]:
#| export

class OptimizerBackend:
    """Base class for backend-specific optimizer adapters."""

    @classmethod
    def create(cls, optimizer_cls, params, *args, **kwargs):
        """
        Create a backend-specific optimizer.

        Parameters
        ----------
        optimizer_cls : type
            PyTorch optimizer class.
        params : iterable
            Parameters to optimize.
        *args
            Positional arguments passed to the optimizer.
        **kwargs
            Keyword arguments passed to the optimizer.

        Returns
        -------
        object
            Backend-specific optimizer.
        """
        raise NotImplementedError

In [ ]:
#| export

@register_optimizer_backend("torch")
class TorchOptimizerBackend(OptimizerBackend):
    """
    Backend adapter for PyTorch optimizers.

    This is the native optimizer backend. It instantiates the supplied
    PyTorch optimizer class directly and returns the resulting optimizer
    instance.

    Parameters
    ----------
    optimizer_cls : type
        PyTorch optimizer class to instantiate.
    params : iterable
        Parameters or parameter groups to optimize.
    *args
        Positional arguments passed to the optimizer.
    **kwargs
        Keyword arguments passed to the optimizer.

    Returns
    -------
    torch.optim.Optimizer
        Instantiated PyTorch optimizer.
    """

    @classmethod
    def create(cls, optimizer_cls, params, *args, **kwargs):
        """Instantiate and return the PyTorch optimizer."""
        return optimizer_cls(params, *args, **kwargs)


@register_optimizer_backend("fastai")
class FastaiOptimizerBackend(OptimizerBackend):
    """
    Backend adapter for fastai optimizers.

    The optimizer is first instantiated using its native PyTorch
    implementation and then wrapped with fastai's ``OptimWrapper``.
    This allows fastai training components to use the same PyTorch
    optimizer while exposing the interface expected by fastai.

    Parameters
    ----------
    optimizer_cls : type
        PyTorch optimizer class to instantiate.
    params : iterable
        Parameters or parameter groups to optimize.
    *args
        Positional arguments passed to the PyTorch optimizer.
    **kwargs
        Keyword arguments passed to the optimizer.

    Returns
    -------
    fastai.optimizer.OptimWrapper
        Fastai wrapper around the PyTorch optimizer.
    """

    @classmethod
    def create(cls, optimizer_cls, params, *args, **kwargs):
        """Instantiate a PyTorch optimizer and wrap it for fastai."""
        optimizer = optimizer_cls(params, *args, **kwargs)
        return OptimWrapper(optimizer)


@register_optimizer_backend("monai")
class MonaiOptimizerBackend(OptimizerBackend):
    """
    Backend adapter for MONAI optimizers.

    MONAI training workflows use PyTorch optimizers, so this adapter
    currently instantiates and returns the supplied PyTorch optimizer
    directly.

    Parameters
    ----------
    optimizer_cls : type
        PyTorch optimizer class to instantiate.
    params : iterable
        Parameters or parameter groups to optimize.
    *args
        Positional arguments passed to the optimizer.
    **kwargs
        Keyword arguments passed to the optimizer.

    Returns
    -------
    torch.optim.Optimizer
        Instantiated PyTorch optimizer.
    """

    @classmethod
    def create(cls, optimizer_cls, params, *args, **kwargs):
        """Instantiate and return the PyTorch optimizer for MONAI."""
        return optimizer_cls(params, *args, **kwargs)


@register_optimizer_backend("ignite")
class IgniteOptimizerBackend(OptimizerBackend):
    """
    Backend adapter for Ignite optimizers.

    Ignite training engines can use standard PyTorch optimizers.
    This adapter therefore currently returns the supplied PyTorch
    optimizer without additional wrapping.

    Parameters
    ----------
    optimizer_cls : type
        PyTorch optimizer class to instantiate.
    params : iterable
        Parameters or parameter groups to optimize.
    *args
        Positional arguments passed to the optimizer.
    **kwargs
        Keyword arguments passed to the optimizer.

    Returns
    -------
    torch.optim.Optimizer
        Instantiated PyTorch optimizer.
    """

    @classmethod
    def create(cls, optimizer_cls, params, *args, **kwargs):
        """Instantiate and return the PyTorch optimizer for Ignite."""
        return optimizer_cls(params, *args, **kwargs)


@register_optimizer_backend("keras")
class KerasOptimizerBackend(OptimizerBackend):
    """
    Backend adapter for Keras optimizers.

    This adapter currently uses the PyTorch optimizer implementation,
    matching the temporary backend strategy used by bioMONAI. It can
    later be replaced with native Keras optimizer construction without
    changing the public ``BioOptimizer`` interface.

    Parameters
    ----------
    optimizer_cls : type
        PyTorch optimizer class to instantiate.
    params : iterable
        Parameters or parameter groups to optimize.
    *args
        Positional arguments passed to the optimizer.
    **kwargs
        Keyword arguments passed to the optimizer.

    Returns
    -------
    torch.optim.Optimizer
        Instantiated PyTorch optimizer.
    """

    @classmethod
    def create(cls, optimizer_cls, params, *args, **kwargs):
        """Instantiate and return the PyTorch optimizer for Keras."""
        return optimizer_cls(params, *args, **kwargs)

### Base function

In [ ]:
#| export

class BioOptimizer:
    """
    Backend-independent wrapper for PyTorch optimizers.

    Each optimizer subclass defines the corresponding PyTorch optimizer
    through the ``_default`` class attribute. The selected backend then
    determines how that optimizer is constructed.

    Parameters
    ----------
    params : iterable
        Iterable of parameters to optimize.
    backend : str, default="torch"
        Backend used to construct the optimizer.
    *args
        Positional arguments passed to the optimizer.
    **kwargs
        Keyword arguments passed to the optimizer.

    Attributes
    ----------
    backend : str
        Backend used by the optimizer.
    params : iterable
        Parameters passed to the optimizer.
    optimizer : object
        Backend-specific optimizer instance.

    Examples
    --------
    >>> optimizer = Adam(model.parameters(), lr=1e-3)
    >>> optimizer = Adam(model.parameters(), lr=1e-3, backend="fastai")
    """

    _default = None

    @classmethod
    def _get_optimizer(cls, backend):
        """Return the optimizer backend adapter."""
        if backend not in OPTIMIZER_BACKENDS:
            raise ValueError(
                f"Unknown optimizer backend: {backend!r}. "
                f"Available backends: {list(OPTIMIZER_BACKENDS)}"
            )
        return OPTIMIZER_BACKENDS[backend]

    def __init__(
        self,
        params,
        *args,
        backend="torch",
        **kwargs,
    ):
        self.backend = backend
        self.params = params

        optimizer_cls = self._get_optimizer_cls()
        backend_cls = self._get_optimizer(backend)

        self.optimizer = backend_cls.create(
            optimizer_cls,
            params,
            *args,
            **kwargs,
        )

    @classmethod
    def _get_optimizer_cls(cls):
        """Return the underlying PyTorch optimizer class."""
        if cls._default is None:
            raise NotImplementedError(
                f"{cls.__name__} must define '_default'."
            )
        return cls._default

## Optimizers

In [ ]:
#| export

class Adadelta(BioOptimizer):
    """
    Adaptive learning-rate optimizer that restricts aggressive updates
    by using a running window of gradient updates.

    Adadelta is designed to reduce the need to manually select a global
    learning rate and is an extension of Adagrad that avoids continually
    accumulating all past squared gradients.

    Parameters and behavior are provided by ``torch.optim.Adadelta``.
    """

    _default = toptim.Adadelta


class Adafactor(BioOptimizer):
    """
    Memory-efficient adaptive optimizer designed for large models.

    Adafactor reduces optimizer-state memory requirements by factorizing
    the second-moment estimates for suitable parameter tensors. It is
    particularly useful when the optimizer state would otherwise consume
    substantial memory.

    Parameters and behavior are provided by ``torch.optim.Adafactor``.
    """

    _default = toptim.Adafactor


class Adagrad(BioOptimizer):
    """
    Adaptive gradient optimizer that maintains a separate learning rate
    for each parameter.

    Adagrad accumulates the squared gradients over time and uses this
    accumulated information to scale parameter updates. It can be
    useful when parameters have gradients with substantially different
    frequencies.

    Parameters and behavior are provided by ``torch.optim.Adagrad``.
    """

    _default = toptim.Adagrad


class Adam(BioOptimizer):
    """
    Adaptive optimizer combining momentum-like first-moment estimates
    with second-moment estimates of the gradients.

    Adam is widely used for deep learning because it adapts the learning
    rate independently for each parameter while incorporating estimates
    of both the gradient and its squared magnitude.

    Parameters and behavior are provided by ``torch.optim.Adam``.
    """

    _default = toptim.Adam


class AdamW(BioOptimizer):
    """
    Adam optimizer with decoupled weight decay.

    AdamW separates weight decay from the adaptive gradient update,
    making the weight-decay coefficient independent of the gradient
    normalization performed by Adam.

    Parameters and behavior are provided by ``torch.optim.AdamW``.
    """

    _default = toptim.AdamW


class Adamax(BioOptimizer):
    """
    Adam variant based on the infinity norm of the gradients.

    Adamax replaces Adam's second-moment estimate with an exponentially
    weighted infinity-norm estimate. It can provide a different
    numerical behavior from standard Adam, particularly for gradients
    with large or highly variable magnitudes.

    Parameters and behavior are provided by ``torch.optim.Adamax``.
    """

    _default = toptim.Adamax


class ASGD(BioOptimizer):
    """
    Averaged Stochastic Gradient Descent optimizer.

    ASGD maintains a running average of parameter values and can use
    this averaged solution to improve convergence and generalization
    in suitable optimization problems.

    Parameters and behavior are provided by ``torch.optim.ASGD``.
    """

    _default = toptim.ASGD


class LBFGS(BioOptimizer):
    """
    Limited-memory BFGS optimizer.

    LBFGS is a quasi-Newton optimization method that approximates
    second-order information without explicitly storing the full
    Hessian. It can be effective for smaller problems but generally
    requires substantially more memory and computation per optimization
    step than first-order optimizers.

    LBFGS requires a closure that recomputes the model output and loss
    when ``step`` is called.

    Parameters and behavior are provided by ``torch.optim.LBFGS``.
    """

    _default = toptim.LBFGS


class NAdam(BioOptimizer):
    """
    Adam optimizer with Nesterov momentum.

    NAdam combines Adam's adaptive first- and second-moment estimates
    with a Nesterov-style momentum formulation, allowing the gradient
    update to incorporate a look-ahead component.

    Parameters and behavior are provided by ``torch.optim.NAdam``.
    """

    _default = toptim.NAdam


class RAdam(BioOptimizer):
    """
    Rectified Adam optimizer.

    RAdam rectifies the adaptive learning rate during the early stages
    of optimization to account for the variance of the adaptive
    momentum estimate.

    Parameters and behavior are provided by ``torch.optim.RAdam``.
    """

    _default = toptim.RAdam


class RMSprop(BioOptimizer):
    """
    Root Mean Square Propagation optimizer.

    RMSprop scales parameter updates using an exponentially weighted
    average of squared gradients. This helps adapt the effective
    learning rate independently for each parameter.

    Parameters and behavior are provided by ``torch.optim.RMSprop``.
    """

    _default = toptim.RMSprop


class Rprop(BioOptimizer):
    """
    Resilient backpropagation optimizer.

    Rprop adapts the update magnitude for each parameter based primarily
    on changes in the sign of its gradient rather than the gradient's
    absolute magnitude.

    Parameters and behavior are provided by ``torch.optim.Rprop``.
    """

    _default = toptim.Rprop


class SGD(BioOptimizer):
    """
    Stochastic Gradient Descent optimizer.

    SGD updates parameters using their gradients and optionally supports
    momentum, dampening, weight decay, and Nesterov momentum.

    Parameters and behavior are provided by ``torch.optim.SGD``.
    """

    _default = toptim.SGD


class SparseAdam(BioOptimizer):
    """
    Adam variant designed for parameters with sparse gradients.

    SparseAdam maintains adaptive first- and second-moment estimates
    while updating only the entries associated with sparse gradients.
    It is intended for models and parameters that produce sparse
    gradients, such as suitable embedding layers.

    Parameters and behavior are provided by ``torch.optim.SparseAdam``.
    """

    _default = toptim.SparseAdam

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()